In [ ]:
import sys, os
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

# Fusão rápida: séries temporais + manchetes verificadas

**Demonstração, não prova de alfa.** Execute no Google Colab com GPU T4 ou L4. Este notebook clona ou atualiza o repositório e lê apenas `MyDrive/latent_fusion_data/fusion_event_panel.npz` e `fusion_events.jsonl.gz`. Recalcula nesta execução **todos** os embeddings numéricos com MOMENT-1-small a partir dos retornos já conhecidos e **todos** os embeddings textuais das manchetes verificadas com MiniLM; não lê vetores antigos do painel nem `article_embedding_cache`. Sem Qwen e sem corpos de artigos. Os resultados vão para uma pasta separada no Drive.

A data `decision_date` da manchete já está atrasada em um pregão. O universo fixo é ADBE, BABA, COST, CVX, GILD, NKE, PYPL e QCOM; o alvo é o retorno futuro de cinco pregões do ativo menos a média deste universo. Treino, validação e teste são cronológicos, com purga de rótulos e janela inicial completa. O teste 2022–23 **já foi inspecionado**: estes gráficos não representam confirmação independente.

In [ ]:
import importlib.util
import subprocess
from importlib.metadata import PackageNotFoundError, version

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('Execute exclusivamente no Google Colab com GPU') from exc
drive.mount('/content/drive')
REPO = Path('/content/latent_fusion')
if REPO.exists():
    if not (REPO / '.git').is_dir():
        raise RuntimeError(f'Pasta existente não é repositório Git: {REPO}')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/rbfcdog/latent_fusion.git', str(REPO)], check=True)
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('PyTorch indisponível; selecione uma GPU no ambiente de execução')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Ative uma GPU T4 ou L4: Ambiente de execução → Alterar tipo de ambiente')
dependencies = []
if importlib.util.find_spec('momentfm') is None:
    dependencies.append('momentfm')
if importlib.util.find_spec('sentence_transformers') is None:
    dependencies.append('sentence-transformers')
try:
    transformers_major = int(version('transformers').split('.')[0])
except PackageNotFoundError:
    transformers_major = 0
if not transformers_major or transformers_major >= 5:
    dependencies.append('transformers<5')
if dependencies:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *dependencies, 'transformers<5'], check=True)
DATA = Path('/content/drive/MyDrive/latent_fusion_data')
PANEL, NEWS = DATA / 'fusion_event_panel.npz', DATA / 'fusion_events.jsonl.gz'
for file in (PANEL, NEWS):
    if not file.is_file():
        raise FileNotFoundError(f'Envie o artefato auditado para {file}')
OUT = DATA / 'fast_fusion_demo_results'
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))
{'gpu': torch.cuda.get_device_name(0), 'painel': str(PANEL), 'notícias': str(NEWS), 'saída': str(OUT)}

## Geração de embeddings, treino e comparação

A preparação gera embeddings frescos para ambas as modalidades **antes** do ajuste. A cabeça numérica e a fusão com atenção cruzada usam as mesmas janelas de MOMENT, linhas, datas, alvo e parada antecipada pela validação (máximo de oito épocas, paciência de duas). A fusão usa até oito manchetes dos últimos cinco pregões, idade em sessões, novidade causal e 17 indicadores léxicos; sem manchete recente, segue seu próprio ramo numérico. O treino e todos os forwards ocorrem somente na GPU do Colab.

In [ ]:
from src.fusion.fast_demo_experiment import run_experiment
result = run_experiment(PANEL, NEWS, OUT, cost_bps=10)
{'arquivos': str(result['output']), 'históricos': [str(OUT / f'{model}_history.csv') for model in result['histories']]}

## Resultados fora da amostra

O erro MSE/MAE e a correlação IC usam **todos os pregões** da amostra, inclusive dias sem notícia; `mse_com_evento` isola linhas com manchetes recentes. A previsão de excesso zero é referência de erro. As carteiras com pesos positivos são avaliadas após 10 bps por valor negociado (inclusive entrada), contra comprar e manter os mesmos oito ativos a pesos iguais na compra inicial. Alfa e beta usam retornos diários relativos a esse mesmo comprar e manter. Horizontes futuros de cinco pregões se sobrepõem, e o universo foi escolhido ex post: sem inferência de causalidade ou garantia de retornos.

In [ ]:
from IPython.display import Image, display
display(result['signal'], result['portfolio'], result['split_coverage'], result['provenance'])
display(result['coverage'].pivot(index='month', columns='ticker', values='titles'))
display(result['histories']['ts_only'], result['histories']['fusion'])
display(Image(filename=str(OUT / 'fast_fusion_vs_bh.png')))
{'relatório': str(OUT / 'results.md'), 'curvas': str(OUT / 'equity_curves.csv'), 'proveniência': str(OUT / 'run_config.json')}